In [1]:
"""
CNN-resnet18
"""
import time
from datasets import load_dataset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.optim as optim

# Timer start
start_time = time.time()

# Load the dataset
ds = load_dataset("mertcobanov/animals")

# Split the dataset into training and testing sets
ds = ds['train'].train_test_split(test_size=0.2, seed=42)
train_data = ds['train']
test_data = ds['test']

# Define the image transformation pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    transforms.ToTensor(),         # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# Custom dataset class for preprocessing
class CustomDataset(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']
        label = item['label']
        image = self.transform(image)
        return image, label

# Create DataLoader for train and test datasets
train_dataset = CustomDataset(train_data, transform=transform)
test_dataset = CustomDataset(test_data, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Load ResNet-18 model with updated weights parameter
weights = ResNet18_Weights.DEFAULT  # Use default pretrained weights
model = resnet18(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(ds['train'].features['label'].names))  # Replace FC layer to match label count

# Move model to GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training function
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# Testing function
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# Train the model
epochs = 5
for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_accuracy = evaluate(model, test_loader, device)
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss:.4f}, Test Accuracy: {test_accuracy:.2%}")

# Print total execution time
print(f"Total execution time: {time.time() - start_time:.2f} seconds")




Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Epoch 1/5, Loss: 2.7896, Test Accuracy: 78.33%
Epoch 2/5, Loss: 0.9200, Test Accuracy: 86.11%
Epoch 3/5, Loss: 0.3339, Test Accuracy: 89.17%
Epoch 4/5, Loss: 0.1250, Test Accuracy: 90.09%
Epoch 5/5, Loss: 0.0578, Test Accuracy: 89.63%
Total execution time: 1791.82 seconds
